
# ESPAC Livestock Direct Field Emissions (DFE)

This notebook computes exploratory livestock direct field emissions from `outputs/CSVs/02_espac_livestock_lci_table_filtered__summary_<strategy>.csv` and writes:

- `outputs/CSVs/03-05_espac_livestock_lci_table_filtered_dfe__summary_<strategy>.csv`

It is separate from the crop DFE notebook and inherits the livestock summary strategy from the livestock metadata written by notebook 2 when available.








Each cell will show output when executed.3. Run the third cell (livestock DFE processing)2. Run the second cell (load configuration files) 1. Run the first cell (imports and setup)**To run this notebook:****To run this notebook:**
1. Run the first cell (imports and setup)
2. Run the second cell (load configuration files) 
3. Run the third cell (livestock DFE processing)

Each cell will show output when executed.


In [1]:
from pathlib import Path
import re
import json
import unicodedata
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent

DEFAULT_SUMMARY_STRATEGY = 'province'  # fallback only when metadata is missing
LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_filtered_export_summary.json'

# Livestock summary strategy inheritance (similar to crop notebook)
LIVESTOCK_DEFAULT_SUMMARY_STRATEGY = 'province'
LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_livestock_filtered_export_summary.json'
available_livestock_strategy_inputs = sorted(
    (PROJECT_DIR / 'outputs/CSVs').glob('02_espac_livestock_lci_table_filtered__summary_*.csv'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

livestock_summary_meta = {}
if LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH.exists():
    try:
        livestock_summary_meta = json.loads(LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH.read_text(encoding='utf-8')) or {}
    except Exception:
        livestock_summary_meta = {}

_livestock_meta_summary = str(livestock_summary_meta.get('summary_level', '')).strip()
if _livestock_meta_summary:
    LIVESTOCK_SUMMARY_STRATEGY = _livestock_meta_summary
    LIVESTOCK_SUMMARY_SOURCE = 'metadata'
elif available_livestock_strategy_inputs:
    m_latest = re.search(r'__summary_([a-z0-9_]+)$', available_livestock_strategy_inputs[0].stem)
    LIVESTOCK_SUMMARY_STRATEGY = m_latest.group(1) if m_latest else LIVESTOCK_DEFAULT_SUMMARY_STRATEGY
    LIVESTOCK_SUMMARY_SOURCE = 'latest_filtered_csv'
else:
    LIVESTOCK_SUMMARY_STRATEGY = LIVESTOCK_DEFAULT_SUMMARY_STRATEGY
    LIVESTOCK_SUMMARY_SOURCE = 'default'

LIVESTOCK_SUMMARY_TOKEN = re.sub(r"[^a-z0-9]+", "_", str(LIVESTOCK_SUMMARY_STRATEGY).strip().lower()).strip("_") or 'province'
LIVESTOCK_INPUT_CSV = PROJECT_DIR / f'outputs/CSVs/02_espac_livestock_lci_table_filtered__summary_{LIVESTOCK_SUMMARY_TOKEN}.csv'
LIVESTOCK_INPUT_UNCERTAINTY_CSV = PROJECT_DIR / f'outputs/CSVs/02_espac_livestock_lci_table_filtered__summary_{LIVESTOCK_SUMMARY_TOKEN}_uncertainty.csv'

print(f'Inherited livestock summary strategy: {LIVESTOCK_SUMMARY_STRATEGY} (source: {LIVESTOCK_SUMMARY_SOURCE})')
print(f'Livestock metadata file: {LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH if LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH.exists() else "(missing)"}')
print(f'Using inherited strategy input CSV: {LIVESTOCK_INPUT_CSV}')
print(f'Using inherited strategy uncertainty CSV: {LIVESTOCK_INPUT_UNCERTAINTY_CSV}')

preferred_input_csv = PROJECT_DIR / f'outputs/CSVs/02_espac_crop_lci_table_filtered__summary_{DEFAULT_SUMMARY_STRATEGY}.csv'
preferred_input_unc_csv = PROJECT_DIR / f'outputs/CSVs/02_espac_crop_lci_table_filtered__summary_{DEFAULT_SUMMARY_STRATEGY}_uncertainty.csv'
available_strategy_inputs = sorted((PROJECT_DIR / 'outputs/CSVs').glob('02_espac_crop_lci_table_filtered__summary_*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)

INPUT_CSV = preferred_input_csv
INPUT_UNCERTAINTY_CSV = preferred_input_unc_csv
OUTPUT_CSV = PROJECT_DIR / f'outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_{DEFAULT_SUMMARY_STRATEGY}.csv'
OUTPUT_UNCERTAINTY_CSV = OUTPUT_CSV.with_name(f"{OUTPUT_CSV.stem}_uncertainty.csv")

FACTORS_YML = PROJECT_DIR / 'inputs/03-05_dfe_factors.yml'
FACTORS_XLSX = PROJECT_DIR / 'inputs/03_Factors.xlsx'
NUTRIENT_CONTENTS_YML = PROJECT_DIR / 'inputs/03_dfe_nutrient_contents.yml'
REGENERATE_YAML_FROM_XLSX = False

assert FACTORS_YML.exists() or FACTORS_XLSX.exists(), 'Need inputs/03-05_dfe_factors.yml or inputs/03_Factors.xlsx'
assert NUTRIENT_CONTENTS_YML.exists(), f'Missing nutrient-content YAML: {NUTRIENT_CONTENTS_YML}'

print('Configuration files loaded successfully.')
print(f'Factors YAML: {FACTORS_YML}')
print(f'Nutrient contents YAML: {NUTRIENT_CONTENTS_YML}')



Inherited livestock summary strategy: product (source: metadata)
Livestock metadata file: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\02_latest_livestock_filtered_export_summary.json
Using inherited strategy input CSV: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_livestock_lci_table_filtered__summary_product.csv
Using inherited strategy uncertainty CSV: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_livestock_lci_table_filtered__summary_product_uncertainty.csv
Configuration files loaded successfully.
Factors YAML: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03-05_dfe_factors.yml
Nutrient contents YAML: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03_dfe_nutrient_contents.yml


In [2]:
def export_factors_xlsx_to_yaml(xlsx_path: Path, yml_path: Path) -> None:
    import pandas as pd

    factors_df = pd.read_excel(xlsx_path, sheet_name='factors')
    refs_df = pd.read_excel(xlsx_path, sheet_name='references')
    factors_df = factors_df.rename(columns={c: str(c).strip() for c in factors_df.columns})

    factor_values = {}
    factor_details = {}
    for _, row in factors_df.iterrows():
        var = str(row.get('Var', '')).strip()
        if not var or var.lower() == 'nan':
            continue
        val = float(row.get('Val', 0) or 0)
        factor_values[var] = val
        detail = {'value': val}
        if 'Comment' in factors_df.columns and pd.notna(row.get('Comment')):
            detail['comment'] = str(row.get('Comment'))
        if 'Current value' in factors_df.columns and pd.notna(row.get('Current value')):
            detail['current_value'] = str(row.get('Current value'))
        factor_details[var] = detail

    if yml_path.exists():
        with yml_path.open('r', encoding='utf-8') as f:
            existing = yaml.safe_load(f) or {}
    else:
        existing = {}

    existing.setdefault('meta', {})
    existing['meta']['source_excel'] = str(xlsx_path).replace('\\', '/')
    existing['meta']['source_sheet'] = 'factors'
    existing['knime_factors'] = factor_values
    existing['knime_factor_details'] = factor_details
    existing['references_sheet_preview'] = refs_df.fillna('').astype(str).head(10).to_dict(orient='records')

    with yml_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(existing, f, sort_keys=False, allow_unicode=True)


if REGENERATE_YAML_FROM_XLSX:
    export_factors_xlsx_to_yaml(FACTORS_XLSX, FACTORS_YML)
    print(f'Regenerated {FACTORS_YML} from {FACTORS_XLSX}')

with FACTORS_YML.open('r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f) or {}

KNIME_FACTORS = CFG.get('knime_factors', {})
ASSUMPTIONS = CFG.get('assumptions', {})
print('Loaded factors:', len(KNIME_FACTORS), 'from', FACTORS_YML)

with NUTRIENT_CONTENTS_YML.open('r', encoding='utf-8') as f:
    NUTRIENT_CFG = yaml.safe_load(f) or {}

NUTRIENT_MAPS = (NUTRIENT_CFG.get('espac_default_mappings', {}) or {})
print('Loaded nutrient-content mappings from', NUTRIENT_CONTENTS_YML)


Loaded factors: 52 from c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03-05_dfe_factors.yml
Loaded nutrient-content mappings from c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03_dfe_nutrient_contents.yml


In [3]:
def _norm_text(x: str) -> str:
    s = str(x).strip().upper()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r'\s+', ' ', s)
    return s


def to_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')


def get_factor(name: str, default: float = 0.0) -> float:
    try:
        return float(KNIME_FACTORS.get(name, default))
    except Exception:
        return float(default)


def infer_crop_type(crop: str, category: str) -> str:
    txt = _norm_text(crop)
    groups = (ASSUMPTIONS.get('crop_type_inference', {}) or {}).get('keyword_groups', {}) or {}
    for crop_type, keywords in groups.items():
        for kw in keywords or []:
            if _norm_text(kw) in txt:
                return crop_type
    fallback = ((ASSUMPTIONS.get('crop_type_inference', {}) or {}).get('fallback_by_category', {}) or {})
    return str(fallback.get(str(category).strip().lower(), 'cereal'))


def ensure_numeric_columns(df: pd.DataFrame, columns) -> pd.DataFrame:
    out = df.copy()
    for c in columns:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = to_num(out[c]).fillna(0.0)
    return out


def apply_fraction_columns(df: pd.DataFrame, source_col: str, frac_cfg: Dict[str, float], prefix: str):
    amount = to_num(df.get(source_col, 0)).fillna(0.0)
    n = amount * float(frac_cfg.get('N', 0.0))
    p = amount * float(frac_cfg.get('P', 0.0))
    k = amount * float(frac_cfg.get('K', 0.0))
    return {
        f'{prefix}_N': n,
        f'{prefix}_P': p,
        f'{prefix}_K': k,
    }


def build_min_max_scenarios(df_mid: pd.DataFrame, df_unc: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df_min = df_mid.copy()
    df_max = df_mid.copy()
    for c in df_mid.columns:
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'
        if cmin in df_unc.columns:
            df_min[c] = pd.to_numeric(df_unc[cmin], errors='coerce').fillna(pd.to_numeric(df_min[c], errors='coerce'))
        if cmax in df_unc.columns:
            df_max[c] = pd.to_numeric(df_unc[cmax], errors='coerce').fillna(pd.to_numeric(df_max[c], errors='coerce'))
    return df_min, df_max


def build_uncertainty_output(df_out: pd.DataFrame, df_unc_in: pd.DataFrame, df_min_out: pd.DataFrame, df_max_out: pd.DataFrame) -> pd.DataFrame:
    key_cols = [c for c in ['Region', 'Province', 'Crop', 'Category'] if c in df_out.columns]
    unc_base = df_out[key_cols].copy() if key_cols else pd.DataFrame(index=df_out.index)
    unc_data = {}

    for c in df_out.columns:
        if not pd.api.types.is_numeric_dtype(df_out[c]):
            continue
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'

        if cmin in df_unc_in.columns and cmax in df_unc_in.columns:
            lo = pd.to_numeric(df_unc_in[cmin], errors='coerce')
            hi = pd.to_numeric(df_unc_in[cmax], errors='coerce')
        else:
            lo = pd.to_numeric(df_min_out.get(c, df_out[c]), errors='coerce')
            hi = pd.to_numeric(df_max_out.get(c, df_out[c]), errors='coerce')

        mid = pd.to_numeric(df_out[c], errors='coerce')
        lo2 = np.minimum(np.minimum(lo, hi), mid)
        hi2 = np.maximum(np.maximum(lo, hi), mid)
        unc_data[cmin] = pd.Series(lo2, index=df_out.index).fillna(mid)
        unc_data[cmax] = pd.Series(hi2, index=df_out.index).fillna(mid)

    unc_values = pd.DataFrame(unc_data, index=df_out.index)
    return pd.concat([unc_base, unc_values], axis=1).copy()


def apply_strategy_specific_export_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if SUMMARY_TOKEN == 'crop_group_national' and 'Crop_group' in out.columns:
        out['Category'] = out['Crop_group'].fillna('').astype(str)
    return out



## Livestock DFE workflow

This section is fully separate from the crop DFE workflow.

It inherits the latest livestock filtered-export metadata from notebook 2 when available, reads the corresponding filtered livestock CSV plus its uncertainty file, and computes exploratory annual livestock emissions for:

- enteric fermentation CH4;
- manure-management CH4;
- manure-management NH3 and NOx (EMEP/EEA 2023 Tier 1);
- manure-management direct and indirect N2O-N / N2O (IPCC 2019 Tier 1);
- managed manure N remaining after manure-management losses.

If no livestock filtered CSV has been exported yet from notebook 2, this section will report that state and skip without affecting the crop workflow.


In [4]:

from typing import Any, Optional

print('=== Starting Livestock DFE Processing ===')

LIVESTOCK_DEFAULT_SUMMARY_STRATEGY = 'province'
LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_livestock_filtered_export_summary.json'
available_livestock_strategy_inputs = sorted(
    (PROJECT_DIR / 'outputs/CSVs').glob('02_espac_livestock_lci_table_filtered__summary_*.csv'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

livestock_summary_meta = {}
if LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH.exists():
    try:
        livestock_summary_meta = json.loads(LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH.read_text(encoding='utf-8')) or {}
    except Exception:
        livestock_summary_meta = {}

_livestock_meta_summary = str(livestock_summary_meta.get('summary_level', '')).strip()
if _livestock_meta_summary:
    LIVESTOCK_SUMMARY_STRATEGY = _livestock_meta_summary
    LIVESTOCK_SUMMARY_SOURCE = 'metadata'
elif available_livestock_strategy_inputs:
    m_latest = re.search(r'__summary_([a-z0-9_]+)$', available_livestock_strategy_inputs[0].stem)
    LIVESTOCK_SUMMARY_STRATEGY = m_latest.group(1) if m_latest else LIVESTOCK_DEFAULT_SUMMARY_STRATEGY
    LIVESTOCK_SUMMARY_SOURCE = 'latest_filtered_csv'
else:
    LIVESTOCK_SUMMARY_STRATEGY = LIVESTOCK_DEFAULT_SUMMARY_STRATEGY
    LIVESTOCK_SUMMARY_SOURCE = 'default'

LIVESTOCK_SUMMARY_TOKEN = re.sub(r"[^a-z0-9]+", "_", str(LIVESTOCK_SUMMARY_STRATEGY).strip().lower()).strip("_") or 'province'
LIVESTOCK_INPUT_CSV = PROJECT_DIR / f'outputs/CSVs/02_espac_livestock_lci_table_filtered__summary_{LIVESTOCK_SUMMARY_TOKEN}.csv'
LIVESTOCK_INPUT_UNCERTAINTY_CSV = PROJECT_DIR / f'outputs/CSVs/02_espac_livestock_lci_table_filtered__summary_{LIVESTOCK_SUMMARY_TOKEN}_uncertainty.csv'
LIVESTOCK_OUTPUT_CSV = PROJECT_DIR / f'outputs/CSVs/03-05_espac_livestock_lci_table_filtered_dfe__summary_{LIVESTOCK_SUMMARY_TOKEN}.csv'
LIVESTOCK_OUTPUT_UNCERTAINTY_CSV = LIVESTOCK_OUTPUT_CSV.with_name(f"{LIVESTOCK_OUTPUT_CSV.stem}_uncertainty.csv")

LIVESTOCK_DFE_CFG = CFG.get('livestock_dfe', {}) or {}
LIVESTOCK_COMMON_FACTORS = LIVESTOCK_DFE_CFG.get('common_factors', {}) or {}
LIVESTOCK_PROFILE_CFG = LIVESTOCK_DFE_CFG.get('profiles', {}) or {}

print('--- Livestock DFE path discovery ---')
print(f'Inherited livestock summary strategy: {LIVESTOCK_SUMMARY_STRATEGY} (source: {LIVESTOCK_SUMMARY_SOURCE})')
print(f'Livestock metadata file: {LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH if LIVESTOCK_LATEST_FILTERED_SUMMARY_META_PATH.exists() else "(missing)"}')
print(f'Expected livestock input CSV: {LIVESTOCK_INPUT_CSV}')
print(f'Expected livestock uncertainty CSV: {LIVESTOCK_INPUT_UNCERTAINTY_CSV}')

LIVESTOCK_READY = LIVESTOCK_INPUT_CSV.exists() and LIVESTOCK_INPUT_UNCERTAINTY_CSV.exists()
if not LIVESTOCK_READY:
    print('Livestock filtered exports are not available yet. Run the livestock filter/export block in notebook 2 and click `Export filtered CSV`, then rerun this livestock section.')
else:
    print('Livestock filtered exports detected. Proceeding with livestock DFE workflow.')


def _get_livestock_key_cols(df: pd.DataFrame) -> list[str]:
    return [c for c in ['Region', 'Province', 'Product', 'System'] if c in df.columns]


def build_uncertainty_output_for_keys(
    df_out: pd.DataFrame,
    df_unc_in: pd.DataFrame,
    df_min_out: pd.DataFrame,
    df_max_out: pd.DataFrame,
    key_cols: list[str],
) -> pd.DataFrame:
    unc_base = df_out[key_cols].copy() if key_cols else pd.DataFrame(index=df_out.index)
    unc_data = {}

    for c in df_out.columns:
        if not pd.api.types.is_numeric_dtype(df_out[c]):
            continue
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'

        if cmin in df_unc_in.columns and cmax in df_unc_in.columns:
            lo = pd.to_numeric(df_unc_in[cmin], errors='coerce')
            hi = pd.to_numeric(df_unc_in[cmax], errors='coerce')
        else:
            lo = pd.to_numeric(df_min_out.get(c, df_out[c]), errors='coerce')
            hi = pd.to_numeric(df_max_out.get(c, df_out[c]), errors='coerce')

        mid = pd.to_numeric(df_out[c], errors='coerce')
        lo2 = np.minimum(np.minimum(lo, hi), mid)
        hi2 = np.maximum(np.maximum(lo, hi), mid)
        unc_data[cmin] = pd.Series(lo2, index=df_out.index).fillna(mid)
        unc_data[cmax] = pd.Series(hi2, index=df_out.index).fillna(mid)

    return pd.concat([unc_base, pd.DataFrame(unc_data, index=df_out.index)], axis=1).copy()


def _safe_float(x: Any, default: float = 0.0) -> float:
    try:
        if pd.isna(x):
            return float(default)
        return float(x)
    except Exception:
        return float(default)


def _resolve_livestock_climate_zone(region: Any) -> str:
    climate_map = (((LIVESTOCK_COMMON_FACTORS.get('ipcc_2019') or {}).get('climate_zone_by_region', {}) or {}))
    return str(climate_map.get(str(region), climate_map.get('(unknown)', 'tropical_moist')))


def _resolve_livestock_head_count(df: pd.DataFrame) -> pd.Series:
    animals_total = pd.to_numeric(df.get('Animals_total_head'), errors='coerce') if 'Animals_total_head' in df.columns else pd.Series(np.nan, index=df.index)
    producing = pd.to_numeric(df.get('Producing_animals_head'), errors='coerce') if 'Producing_animals_head' in df.columns else pd.Series(np.nan, index=df.index)
    product = df.get('Product', pd.Series('', index=df.index)).astype(str)
    return pd.Series(
        np.where(
            product.eq('eggs'),
            producing.fillna(animals_total),
            animals_total.fillna(producing),
        ),
        index=df.index,
        dtype='float64',
    ).fillna(0.0)


def _resolve_livestock_reference_output_for_dfe(row: pd.Series) -> float:
    ref = pd.to_numeric(pd.Series([row.get('Reference_output_kg_for_dfe')]), errors='coerce').iloc[0]
    if pd.notna(ref) and ref > 0:
        return float(ref)
    ref = pd.to_numeric(pd.Series([row.get('Reference_output_kg')]), errors='coerce').iloc[0]
    if pd.notna(ref) and ref > 0:
        return float(ref)
    return np.nan


def build_livestock_synthetic_summary_table(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if df.empty:
        return df

    MILK_PRODUCTIVE_LIFE_LACTATIONS = 2.5
    MILK_DAYS_PER_LACTATION = 305.0
    EGGS_PRODUCTIVE_PERIOD_WEEKS = 54.0
    EGGS_PER_HEN_YEAR = 300.0
    EGG_WEIGHT_KG = 0.056699
    EGGS_PRODUCTIVE_DAYS = EGGS_PRODUCTIVE_PERIOD_WEEKS * 7.0
    EGGS_PER_HEN_PRODUCTIVE_LIFE = EGGS_PER_HEN_YEAR * (EGGS_PRODUCTIVE_PERIOD_WEEKS / 52.0)
    EGGS_KG_PER_HEN_PRODUCTIVE_LIFE = EGGS_PER_HEN_PRODUCTIVE_LIFE * EGG_WEIGHT_KG

    ref_output = pd.to_numeric(df.get('Reference_output_kg'), errors='coerce') if 'Reference_output_kg' in df.columns else pd.Series(np.nan, index=df.index)
    head_count = pd.to_numeric(df.get('Animals_emission_basis_head'), errors='coerce') if 'Animals_emission_basis_head' in df.columns else _resolve_livestock_head_count(df)
    scale_to_1kg = 1.0 / ref_output.replace(0, np.nan)
    product = df.get('Product', pd.Series('', index=df.index)).astype(str)
    producing = pd.to_numeric(df.get('Producing_animals_head'), errors='coerce') if 'Producing_animals_head' in df.columns else pd.Series(np.nan, index=df.index)
    milk_kg_head_day = pd.to_numeric(df.get('Milk_output_kg_fpcm_day'), errors='coerce') / producing.replace(0, np.nan) if 'Milk_output_kg_fpcm_day' in df.columns else pd.Series(np.nan, index=df.index)
    milk_lifetime_output_kg_head = milk_kg_head_day * MILK_PRODUCTIVE_LIFE_LACTATIONS * MILK_DAYS_PER_LACTATION
    eggs_lifetime_output_kg_head = pd.Series(EGGS_KG_PER_HEN_PRODUCTIVE_LIFE, index=df.index, dtype='float64')
    lifetime_output_kg_head = pd.Series(np.nan, index=df.index, dtype='float64')
    lifetime_output_kg_head.loc[product.eq('milk')] = milk_lifetime_output_kg_head.loc[product.eq('milk')]
    lifetime_output_kg_head.loc[product.eq('eggs')] = eggs_lifetime_output_kg_head.loc[product.eq('eggs')]
    headcount_per_1kg_lifetime = 1.0 / lifetime_output_kg_head.replace(0, np.nan)

    pasture_feed_kg_head_day = pd.to_numeric(df.get('Pasture_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Pasture_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    cultivated_pasture_feed_kg_head_day = pd.to_numeric(df.get('Cultivated_pasture_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Cultivated_pasture_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    natural_pasture_feed_kg_head_day = pd.to_numeric(df.get('Natural_pasture_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Natural_pasture_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    unmatched_pasture_feed_kg_head_day = pd.to_numeric(df.get('Unmatched_pasture_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Unmatched_pasture_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    cultivated_pasture_share_pct = pd.to_numeric(df.get('Cultivated_pasture_feed_share_pct'), errors='coerce') if 'Cultivated_pasture_feed_share_pct' in df.columns else pd.Series(np.nan, index=df.index)
    natural_pasture_share_pct = pd.to_numeric(df.get('Natural_pasture_feed_share_pct'), errors='coerce') if 'Natural_pasture_feed_share_pct' in df.columns else pd.Series(np.nan, index=df.index)
    unmatched_pasture_share_pct = pd.to_numeric(df.get('Unmatched_pasture_feed_share_pct'), errors='coerce') if 'Unmatched_pasture_feed_share_pct' in df.columns else pd.Series(np.nan, index=df.index)
    pasture_linkage_status = df.get('Pasture_linkage_status', pd.Series('', index=df.index)).astype(str) if 'Pasture_linkage_status' in df.columns else pd.Series('', index=df.index)
    supplement_feed_kg_head_day = pd.to_numeric(df.get('Supplement_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Supplement_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    common_feed_kg_head_day = pd.to_numeric(df.get('Common_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Common_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    waste_feed_kg_head_day = pd.to_numeric(df.get('Waste_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Waste_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)

    proxy_forage_kg_head_day = pd.to_numeric(df.get('Proxy_forage_kg_head_day'), errors='coerce').fillna(0.0) if 'Proxy_forage_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    proxy_tree_forage_kg_head_day = pd.to_numeric(df.get('Proxy_tree_forage_kg_head_day'), errors='coerce').fillna(0.0) if 'Proxy_tree_forage_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    proxy_compound_feed_kg_head_day = pd.to_numeric(df.get('Proxy_compound_feed_kg_head_day'), errors='coerce').fillna(0.0) if 'Proxy_compound_feed_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    proxy_mineral_kg_head_day = pd.to_numeric(df.get('Proxy_mineral_kg_head_day'), errors='coerce').fillna(0.0) if 'Proxy_mineral_kg_head_day' in df.columns else pd.Series(0.0, index=df.index)
    water_l_head_day = pd.to_numeric(df.get('Water_l_head_day'), errors='coerce').fillna(0.0) if 'Water_l_head_day' in df.columns else pd.Series(0.0, index=df.index)
    electricity_kwh_head_day = pd.to_numeric(df.get('Electricity_kWh_head_day'), errors='coerce').fillna(0.0) if 'Electricity_kWh_head_day' in df.columns else pd.Series(0.0, index=df.index)

    lifetime_days = pd.Series(1.0, index=df.index, dtype='float64')
    lifetime_days.loc[product.eq('milk')] = MILK_PRODUCTIVE_LIFE_LACTATIONS * MILK_DAYS_PER_LACTATION
    lifetime_days.loc[product.eq('eggs')] = EGGS_PRODUCTIVE_DAYS
    default_feed_factor = head_count * scale_to_1kg
    lifetime_feed_factor = lifetime_days * headcount_per_1kg_lifetime
    feed_factor = default_feed_factor.copy()
    feed_factor.loc[product.eq('milk') | product.eq('eggs')] = lifetime_feed_factor.loc[product.eq('milk') | product.eq('eggs')]
    df['Synthetic_pasture_feed_total_kg_per_1kg_product'] = (pasture_feed_kg_head_day + proxy_forage_kg_head_day + proxy_tree_forage_kg_head_day) * feed_factor
    df['Synthetic_cultivated_pasture_feed_total_kg_per_1kg_product'] = cultivated_pasture_feed_kg_head_day * feed_factor
    df['Synthetic_natural_pasture_feed_total_kg_per_1kg_product'] = natural_pasture_feed_kg_head_day * feed_factor
    df['Synthetic_unmatched_pasture_feed_total_kg_per_1kg_product'] = unmatched_pasture_feed_kg_head_day * feed_factor
    df['Synthetic_proxy_forage_feed_total_kg_per_1kg_product'] = (proxy_forage_kg_head_day + proxy_tree_forage_kg_head_day) * feed_factor
    df['Synthetic_cultivated_pasture_feed_share_pct'] = cultivated_pasture_share_pct
    df['Synthetic_natural_pasture_feed_share_pct'] = natural_pasture_share_pct
    df['Synthetic_unmatched_pasture_feed_share_pct'] = unmatched_pasture_share_pct
    df['Synthetic_pasture_linkage_status'] = pasture_linkage_status
    df['Synthetic_supplement_feed_total_kg_per_1kg_product'] = (supplement_feed_kg_head_day + proxy_mineral_kg_head_day) * feed_factor
    df['Synthetic_common_feed_total_kg_per_1kg_product'] = (common_feed_kg_head_day + proxy_compound_feed_kg_head_day) * feed_factor
    df['Synthetic_waste_feed_total_kg_per_1kg_product'] = waste_feed_kg_head_day * feed_factor
    df['Synthetic_water_total_L_per_1kg_product'] = water_l_head_day * feed_factor
    df['Synthetic_electricity_total_kWh_per_1kg_product'] = electricity_kwh_head_day * feed_factor
    df['Synthetic_headcount_per_1kg_product'] = head_count * scale_to_1kg
    df.loc[product.eq('milk') | product.eq('eggs'), 'Synthetic_headcount_per_1kg_product'] = headcount_per_1kg_lifetime.loc[product.eq('milk') | product.eq('eggs')]
    df['Synthetic_lifetime_output_kg_head'] = lifetime_output_kg_head
    df['Synthetic_normalized_product_output_kg'] = np.where(scale_to_1kg.notna(), 1.0, np.nan)
    df['Synthetic_feed_mapping_note'] = 'pasture total = ESPAC pasture + proxy forage + proxy tree forage; cultivated and natural pasture totals show only the ESPAC-linked split of cattle pasture, while unmatched pasture is the residual cattle pasture share not linked to pcnac or sunac; supplement = ESPAC supplement + proxy mineral; common = ESPAC common feed + proxy compound feed; waste = ESPAC waste only'
    df['Synthetic_headcount_basis_note'] = np.select(
        [product.eq('milk'), product.eq('eggs')],
        [
            f'Milk headcount per kg is based on lifetime FPCM per cow using {MILK_PRODUCTIVE_LIFE_LACTATIONS:.1f} lactations of {MILK_DAYS_PER_LACTATION:.0f} days each.',
            f'Egg headcount per kg is based on one laying cycle of {EGGS_PRODUCTIVE_PERIOD_WEEKS:.0f} weeks and {EGGS_PER_HEN_YEAR:.0f} eggs/hen/year, converted with {EGG_WEIGHT_KG:.6f} kg/egg.',
        ],
        default='Headcount per kg is based on current inventory mass basis for live weight equivalent products.',
    )

    display_cols = [
        c for c in [
            'Product', 'Species', 'System', 'Region', 'Province', 'Functional_unit', 'Infrastructure_type', 'Infrastructure_source', 'Infra_water_electricity_status',
            'Water_l_head_day', 'Electricity_kWh_head_day', 'Synthetic_normalized_product_output_kg',
            'Synthetic_lifetime_output_kg_head',
            'Synthetic_headcount_per_1kg_product', 'Synthetic_pasture_feed_total_kg_per_1kg_product',
            'Synthetic_cultivated_pasture_feed_total_kg_per_1kg_product', 'Synthetic_natural_pasture_feed_total_kg_per_1kg_product',
            'Synthetic_unmatched_pasture_feed_total_kg_per_1kg_product', 'Synthetic_proxy_forage_feed_total_kg_per_1kg_product',
            'Synthetic_cultivated_pasture_feed_share_pct', 'Synthetic_natural_pasture_feed_share_pct', 'Synthetic_unmatched_pasture_feed_share_pct',
            'Synthetic_pasture_linkage_status', 'Synthetic_supplement_feed_total_kg_per_1kg_product', 'Synthetic_common_feed_total_kg_per_1kg_product',
            'Synthetic_waste_feed_total_kg_per_1kg_product', 'Synthetic_water_total_L_per_1kg_product', 'Synthetic_electricity_total_kWh_per_1kg_product',
            'Synthetic_headcount_basis_note', 'Synthetic_feed_mapping_note', 'Pasture_linkage_note', 'Infra_water_electricity_note', 'Infra_water_electricity_citations'
        ] if c in df.columns
    ]
    return df[display_cols].copy()


def _profile_matches(row: pd.Series, profile_cfg: dict) -> bool:
    applies_to = profile_cfg.get('applies_to', {}) or {}
    for field in ['product', 'system', 'species']:
        allowed = applies_to.get(field)
        if allowed is None:
            continue
        allowed_vals = allowed if isinstance(allowed, list) else [allowed]
        row_val = str(row.get(field.capitalize(), row.get(field, '')))
        if row_val not in [str(v) for v in allowed_vals]:
            return False
    return True


def _resolve_livestock_profile_key(row: pd.Series) -> str:
    for key, cfg in LIVESTOCK_PROFILE_CFG.items():
        if _profile_matches(row, cfg or {}):
            return str(key)
    return ''


def _lookup_manure_ch4_factor(ef_profile_name: str, awms_name: str, climate_zone: str) -> float:
    ef_profiles = (LIVESTOCK_COMMON_FACTORS.get('ipcc_2019_manure_ch4_factors_g_ch4_per_kg_vs') or {})
    profile = ef_profiles.get(ef_profile_name, {}) or {}
    factor_cfg = profile.get(awms_name, {}) or {}
    if not factor_cfg:
        factor_cfg = profile.get('all_systems', {}) or {}
    if not factor_cfg:
        return 0.0
    if climate_zone in factor_cfg:
        return _safe_float(factor_cfg[climate_zone])
    if climate_zone.startswith('tropical') and 'tropical' in factor_cfg:
        return _safe_float(factor_cfg['tropical'])
    for fallback_key in ['tropical_moist', 'tropical_wet', 'tropical_montane', 'tropical_dry', 'tropical']:
        if fallback_key in factor_cfg:
            return _safe_float(factor_cfg[fallback_key])
    first_val = next(iter(factor_cfg.values())) if factor_cfg else 0.0
    return _safe_float(first_val)


def _lookup_awms_n_loss_factors(n_loss_profile: str, awms_name: str) -> dict:
    table = (LIVESTOCK_COMMON_FACTORS.get('ipcc_2019_awms_n_loss_factors') or {})
    return (table.get(n_loss_profile, {}) or {}).get(awms_name, {}) or {}


def compute_livestock_dfe_columns(df_in: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df_in.copy()
    if 'Product' not in df.columns:
        raise ValueError('Livestock DFE input must include `Product`.')
    if 'System' not in df.columns:
        df['System'] = ''
    if 'Species' not in df.columns:
        df['Species'] = ''
    if 'Region' not in df.columns:
        df['Region'] = '(unknown)'

    df['Livestock_profile_key'] = df.apply(_resolve_livestock_profile_key, axis=1)
    df['Livestock_climate_zone'] = df['Region'].map(_resolve_livestock_climate_zone)
    df['Emission_head_basis'] = 'Animals_total_head'
    df.loc[df['Product'].astype(str).eq('eggs'), 'Emission_head_basis'] = 'Producing_animals_head'
    df['Animals_emission_basis_head'] = _resolve_livestock_head_count(df)

    common_ipcc = LIVESTOCK_COMMON_FACTORS.get('ipcc_2019', {}) or {}
    ef4 = _safe_float(common_ipcc.get('indirect_n2o_ef4_volatilisation'), 0.009)
    ef5 = _safe_float(common_ipcc.get('indirect_n2o_ef5_leaching'), 0.011)
    n2_ratio = _safe_float(common_ipcc.get('n2o_to_n2_ratio_for_manure_management'), 3.0)

    out_rows = []
    missing_profiles = []
    for idx, row in df.iterrows():
        row_out = row.copy()
        profile_key = str(row['Livestock_profile_key'])
        profile = LIVESTOCK_PROFILE_CFG.get(profile_key, {}) or {}
        ipcc_profile = profile.get('ipcc_profile', {}) or {}
        emep_profile = profile.get('emep_eea_2023', {}) or {}

        if not profile_key:
            missing_profiles.append(idx)
            for c in [
                'Headcount_emissions_basis_head', 'kgCH4_enteric_yr', 'kgCH4_manure_mgmt_yr',
                'kgNH3_manure_mgmt_total_yr', 'kgNH3_manure_housing_storage_yard_yr', 'kgNH3_manure_application_yr',
                'kgNH3_grazing_yr', 'kgNOx_manure_mgmt_as_NO2_yr', 'kgNexcretion_manure_yr', 'kgVS_manure_yr',
                'kgN2ON_manure_mgmt_direct_yr', 'kgN2ON_manure_mgmt_indirect_yr', 'kgN2ON_manure_mgmt_total_yr',
                'kgN2O_manure_mgmt_direct_yr', 'kgN2O_manure_mgmt_indirect_yr', 'kgN2O_manure_mgmt_total_yr',
                'kgN2_manure_mgmt_yr', 'kgN_manure_after_management_yr', 'kgCH4_livestock_total_yr',
                'FracGasMS_weighted', 'FracLeachMS_weighted', 'EF3_manure_direct_weighted'
            ]:
                row_out[c] = np.nan
            row_out['DFE_livestock_status'] = 'no livestock DFE profile matched in YAML'
            out_rows.append(row_out)
            continue

        head_count = _safe_float(row['Animals_emission_basis_head'])
        nex = _safe_float(ipcc_profile.get('nex_kg_head_yr'))
        vs_day = _safe_float(ipcc_profile.get('vs_kg_head_day'))
        enteric_ef = _safe_float(ipcc_profile.get('enteric_ch4_kg_head_yr'))
        awms_share = ipcc_profile.get('awms_share', {}) or {}
        manure_ch4_profile = str(ipcc_profile.get('manure_ch4_factor_profile', '')).strip()
        n_loss_profile = str(ipcc_profile.get('n_loss_profile', '')).strip()
        climate_zone = str(row['Livestock_climate_zone'])

        weighted_manure_ch4_g_per_kg_vs = 0.0
        weighted_frac_gas = 0.0
        weighted_frac_leach = 0.0
        weighted_ef3 = 0.0
        for awms_name, share in awms_share.items():
            share_f = _safe_float(share)
            weighted_manure_ch4_g_per_kg_vs += share_f * _lookup_manure_ch4_factor(manure_ch4_profile, awms_name, climate_zone)
            n_loss = _lookup_awms_n_loss_factors(n_loss_profile, awms_name)
            weighted_frac_gas += share_f * _safe_float(n_loss.get('frac_gas_ms'))
            weighted_frac_leach += share_f * _safe_float(n_loss.get('frac_leach_ms'))
            weighted_ef3 += share_f * _safe_float(n_loss.get('ef3_direct_n2o'))

        kg_n_manure = head_count * nex
        kg_vs_manure = head_count * vs_day * 365.0
        kg_ch4_enteric = head_count * enteric_ef
        kg_ch4_manure = kg_vs_manure * weighted_manure_ch4_g_per_kg_vs / 1000.0
        kg_n2on_direct = kg_n_manure * weighted_ef3
        kg_n2on_indirect = kg_n_manure * (weighted_frac_gas * ef4 + weighted_frac_leach * ef5)
        kg_n2on_total = kg_n2on_direct + kg_n2on_indirect
        kg_n2o_direct = kg_n2on_direct * 44.0 / 28.0
        kg_n2o_indirect = kg_n2on_indirect * 44.0 / 28.0
        kg_n2o_total = kg_n2on_total * 44.0 / 28.0
        kg_n2 = kg_n_manure * weighted_ef3 * n2_ratio
        frac_loss_total = weighted_frac_gas + weighted_frac_leach + weighted_ef3 + (weighted_ef3 * n2_ratio)
        kg_n_after_mgmt = kg_n_manure * max(0.0, 1.0 - frac_loss_total)

        row_out['Headcount_emissions_basis_head'] = head_count
        row_out['kgCH4_enteric_yr'] = kg_ch4_enteric
        row_out['kgCH4_manure_mgmt_yr'] = kg_ch4_manure
        row_out['kgCH4_livestock_total_yr'] = kg_ch4_enteric + kg_ch4_manure
        row_out['kgNH3_manure_mgmt_total_yr'] = head_count * _safe_float(emep_profile.get('nh3_total_kg_head_yr'))
        row_out['kgNH3_manure_housing_storage_yard_yr'] = head_count * _safe_float(emep_profile.get('nh3_housing_storage_yard_kg_head_yr'))
        row_out['kgNH3_manure_application_yr'] = head_count * _safe_float(emep_profile.get('nh3_manure_application_kg_head_yr'))
        row_out['kgNH3_grazing_yr'] = head_count * _safe_float(emep_profile.get('nh3_grazing_kg_head_yr'))
        row_out['kgNOx_manure_mgmt_as_NO2_yr'] = head_count * _safe_float(emep_profile.get('nox_as_no2_kg_head_yr'))
        row_out['kgNexcretion_manure_yr'] = kg_n_manure
        row_out['kgVS_manure_yr'] = kg_vs_manure
        row_out['FracGasMS_weighted'] = weighted_frac_gas
        row_out['FracLeachMS_weighted'] = weighted_frac_leach
        row_out['EF3_manure_direct_weighted'] = weighted_ef3
        row_out['kgN2ON_manure_mgmt_direct_yr'] = kg_n2on_direct
        row_out['kgN2ON_manure_mgmt_indirect_yr'] = kg_n2on_indirect
        row_out['kgN2ON_manure_mgmt_total_yr'] = kg_n2on_total
        row_out['kgN2O_manure_mgmt_direct_yr'] = kg_n2o_direct
        row_out['kgN2O_manure_mgmt_indirect_yr'] = kg_n2o_indirect
        row_out['kgN2O_manure_mgmt_total_yr'] = kg_n2o_total
        row_out['kgN2_manure_mgmt_yr'] = kg_n2
        row_out['kgN_manure_after_management_yr'] = kg_n_after_mgmt
        ref_output_kg = _resolve_livestock_reference_output_for_dfe(row)
        row_out['Reference_output_kg_dfe_basis'] = ref_output_kg
        row_out['DFE_functional_unit'] = str(row.get('Functional_unit', '1 kg reference product'))
        if pd.notna(ref_output_kg) and ref_output_kg > 0:
            for c in [
                'Headcount_emissions_basis_head', 'kgCH4_enteric_yr', 'kgCH4_manure_mgmt_yr', 'kgCH4_livestock_total_yr',
                'kgNH3_manure_mgmt_total_yr', 'kgNH3_manure_housing_storage_yard_yr', 'kgNH3_manure_application_yr',
                'kgNH3_grazing_yr', 'kgNOx_manure_mgmt_as_NO2_yr', 'kgNexcretion_manure_yr', 'kgVS_manure_yr',
                'kgN2ON_manure_mgmt_direct_yr', 'kgN2ON_manure_mgmt_indirect_yr', 'kgN2ON_manure_mgmt_total_yr',
                'kgN2O_manure_mgmt_direct_yr', 'kgN2O_manure_mgmt_indirect_yr', 'kgN2O_manure_mgmt_total_yr',
                'kgN2_manure_mgmt_yr', 'kgN_manure_after_management_yr'
            ]:
                row_out[c] = _safe_float(row_out.get(c), np.nan) / ref_output_kg
        dfe_alias_map = {
            'Headcount_emissions_basis_head': 'Headcount_emissions_basis_per_1kg_product',
            'kgCH4_enteric_yr': 'kgCH4_enteric_per_1kg_product',
            'kgCH4_manure_mgmt_yr': 'kgCH4_manure_mgmt_per_1kg_product',
            'kgCH4_livestock_total_yr': 'kgCH4_livestock_total_per_1kg_product',
            'kgNH3_manure_mgmt_total_yr': 'kgNH3_manure_mgmt_total_per_1kg_product',
            'kgNH3_manure_housing_storage_yard_yr': 'kgNH3_manure_housing_storage_yard_per_1kg_product',
            'kgNH3_manure_application_yr': 'kgNH3_manure_application_per_1kg_product',
            'kgNH3_grazing_yr': 'kgNH3_grazing_per_1kg_product',
            'kgNOx_manure_mgmt_as_NO2_yr': 'kgNOx_manure_mgmt_as_NO2_per_1kg_product',
            'kgNexcretion_manure_yr': 'kgNexcretion_manure_per_1kg_product',
            'kgVS_manure_yr': 'kgVS_manure_per_1kg_product',
            'kgN2ON_manure_mgmt_direct_yr': 'kgN2ON_manure_mgmt_direct_per_1kg_product',
            'kgN2ON_manure_mgmt_indirect_yr': 'kgN2ON_manure_mgmt_indirect_per_1kg_product',
            'kgN2ON_manure_mgmt_total_yr': 'kgN2ON_manure_mgmt_total_per_1kg_product',
            'kgN2O_manure_mgmt_direct_yr': 'kgN2O_manure_mgmt_direct_per_1kg_product',
            'kgN2O_manure_mgmt_indirect_yr': 'kgN2O_manure_mgmt_indirect_per_1kg_product',
            'kgN2O_manure_mgmt_total_yr': 'kgN2O_manure_mgmt_total_per_1kg_product',
            'kgN2_manure_mgmt_yr': 'kgN2_manure_mgmt_per_1kg_product',
            'kgN_manure_after_management_yr': 'kgN_manure_after_management_per_1kg_product',
        }
        for source_col, alias_col in dfe_alias_map.items():
            row_out[alias_col] = row_out.get(source_col)
        row_out['DFE_livestock_status'] = 'livestock DFE computed from YAML-configured Tier 1 factors'
        out_rows.append(row_out)

    out = pd.DataFrame(out_rows)
    diagnostics = pd.DataFrame({
        'metric': [
            'rows',
            'rows_with_profile',
            'rows_missing_profile',
            'rows_with_positive_enteric_ch4',
            'rows_with_positive_manure_ch4',
            'rows_with_positive_manure_n2o',
        ],
        'value': [
            len(out),
            int(out['Livestock_profile_key'].astype(str).ne('').sum()),
            int(out['Livestock_profile_key'].astype(str).eq('').sum()),
            int((pd.to_numeric(out.get('kgCH4_enteric_yr'), errors='coerce').fillna(0) > 0).sum()),
            int((pd.to_numeric(out.get('kgCH4_manure_mgmt_yr'), errors='coerce').fillna(0) > 0).sum()),
            int((pd.to_numeric(out.get('kgN2O_manure_mgmt_total_yr'), errors='coerce').fillna(0) > 0).sum()),
        ]
    })
    return out, diagnostics


if LIVESTOCK_READY:
    livestock_df_in = pd.read_csv(LIVESTOCK_INPUT_CSV)
    livestock_df_unc_in = pd.read_csv(LIVESTOCK_INPUT_UNCERTAINTY_CSV)
    print('Livestock input shape:', livestock_df_in.shape)
    print('Livestock input uncertainty shape:', livestock_df_unc_in.shape)

    livestock_df_out, livestock_diag = compute_livestock_dfe_columns(livestock_df_in)
    livestock_df_min_in, livestock_df_max_in = build_min_max_scenarios(livestock_df_in, livestock_df_unc_in)
    livestock_df_min_out, _ = compute_livestock_dfe_columns(livestock_df_min_in)
    livestock_df_max_out, _ = compute_livestock_dfe_columns(livestock_df_max_in)

    livestock_key_cols = _get_livestock_key_cols(livestock_df_out)
    livestock_df_unc_out = build_uncertainty_output_for_keys(
        livestock_df_out,
        livestock_df_unc_in,
        livestock_df_min_out,
        livestock_df_max_out,
        key_cols=livestock_key_cols,
    )

    livestock_new_cols = [c for c in livestock_df_out.columns if c not in livestock_df_in.columns]
    livestock_df_out = livestock_df_out[list(livestock_df_in.columns) + livestock_new_cols]

    LIVESTOCK_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    livestock_df_out.to_csv(LIVESTOCK_OUTPUT_CSV, index=False, encoding='utf-8-sig')
    livestock_df_unc_out.to_csv(LIVESTOCK_OUTPUT_UNCERTAINTY_CSV, index=False, encoding='utf-8-sig')
    livestock_synthetic_summary_df = build_livestock_synthetic_summary_table(livestock_df_out)
    LIVESTOCK_SYNTHETIC_SUMMARY_CSV = LIVESTOCK_OUTPUT_CSV.with_name(f"{LIVESTOCK_OUTPUT_CSV.stem}__synthetic_summary.csv")
    livestock_synthetic_summary_df.to_csv(LIVESTOCK_SYNTHETIC_SUMMARY_CSV, index=False, encoding='utf-8-sig')

    print(f'Saved livestock DFE-augmented table to: {LIVESTOCK_OUTPUT_CSV}')
    print(f'Saved livestock DFE uncertainty table to: {LIVESTOCK_OUTPUT_UNCERTAINTY_CSV}')
    print(f'Saved livestock synthetic summary table to: {LIVESTOCK_SYNTHETIC_SUMMARY_CSV}')

    livestock_num_show = [
        'Headcount_emissions_basis_head', 'kgCH4_enteric_yr', 'kgCH4_manure_mgmt_yr', 'kgCH4_livestock_total_yr',
        'kgNH3_manure_mgmt_total_yr', 'kgNOx_manure_mgmt_as_NO2_yr', 'kgN2O_manure_mgmt_total_yr',
        'kgN_manure_after_management_yr'
    ]
    livestock_num_show = [c for c in livestock_num_show if c in livestock_df_out.columns]
    livestock_preview_cols = [c for c in ['Product', 'Species', 'System', 'Region', 'Province', 'Livestock_profile_key', 'Livestock_climate_zone'] if c in livestock_df_out.columns]
    if livestock_num_show:
        display(livestock_df_out[livestock_preview_cols + livestock_num_show].head(20))
        display(livestock_df_out[livestock_num_show].describe().T)
    display(Markdown('### Synthetic livestock summary table'))
    display(livestock_synthetic_summary_df)
    display(livestock_diag)


=== Starting Livestock DFE Processing ===
--- Livestock DFE path discovery ---
Inherited livestock summary strategy: product (source: metadata)
Livestock metadata file: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\02_latest_livestock_filtered_export_summary.json
Expected livestock input CSV: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_livestock_lci_table_filtered__summary_product.csv
Expected livestock uncertainty CSV: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_livestock_lci_table_filtered__summary_product_uncertainty.csv
Livestock filtered exports detected. Proceeding with livestock DFE workflow.
Livestock input shape: (7, 118)
Livestock input uncertainty shape: (7, 195)
Saved livestock DFE-augmented table to: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_livestock_lci_table_filtered_dfe__summary_product.csv
Saved livestock DFE uncertainty table to: c:

,Product,Species,System,Region,Livestock_profile_key,Livestock_climate_zone,Headcount_emissions_basis_head,kgCH4_enteric_yr,kgCH4_manure_mgmt_yr,kgCH4_livestock_total_yr,kgNH3_manure_mgmt_total_yr,kgNOx_manure_mgmt_as_NO2_yr,kgN2O_manure_mgmt_total_yr,kgN_manure_after_management_yr
0,cattle_live,cattle,,(unknown),,tropical_moist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,eggs,chicken,,(unknown),,tropical_moist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,goat_live,caprine,,(unknown),goat_low,tropical_moist,0.094518,0.472590,0.016282,0.488872,0.132325,0.001134,0.002310,0.578939
3,meat_poultry,chicken,,(unknown),,tropical_moist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,milk,cattle,,(unknown),,tropical_moist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ovine_live,ovine,,(unknown),ovine_low,tropical_moist,0.035968,0.179841,0.005508,0.185348,0.050355,0.000432,0.000656,0.164359
6,swine_live,swine,,(unknown),swine_mixed,tropical_moist,0.020140,0.020140,0.137361,0.157501,0.130912,0.000040,0.003539,0.184016


,count,mean,std,min,25%,50%,75%,max
Headcount_emissions_basis_head,3.0,0.050209,0.039180,0.020140,0.028054,0.035968,0.065243,0.094518
kgCH4_enteric_yr,3.0,0.224190,0.229462,0.020140,0.099990,0.179841,0.326215,0.472590
kgCH4_manure_mgmt_yr,3.0,0.053050,0.073214,0.005508,0.010895,0.016282,0.076821,0.137361
kgCH4_livestock_total_yr,3.0,0.277240,0.183806,0.157501,0.171425,0.185348,0.337110,0.488872
kgNH3_manure_mgmt_total_yr,3.0,0.104531,0.046923,0.050355,0.090634,0.130912,0.131618,0.132325
kgNOx_manure_mgmt_as_NO2_yr,3.0,0.000535,0.000554,0.000040,0.000236,0.000432,0.000783,0.001134
kgN2O_manure_mgmt_total_yr,3.0,0.002169,0.001447,0.000656,0.001483,0.002310,0.002925,0.003539
kgN_manure_after_management_yr,3.0,0.309105,0.233890,0.164359,0.174187,0.184016,0.381477,0.578939


### Synthetic livestock summary table

,Product,Species,System,Region,Functional_unit,Infrastructure_type,Infrastructure_source,Infra_water_electricity_status,Water_l_head_day,Electricity_kWh_head_day,...,Synthetic_pasture_linkage_status,Synthetic_supplement_feed_total_kg_per_1kg_product,Synthetic_common_feed_total_kg_per_1kg_product,Synthetic_waste_feed_total_kg_per_1kg_product,Synthetic_water_total_L_per_1kg_product,Synthetic_electricity_total_kWh_per_1kg_product,Synthetic_headcount_basis_note,Synthetic_feed_mapping_note,Infra_water_electricity_note,Infra_water_electricity_citations
0,cattle_live,cattle,,(unknown),1 kg meat (carcass-weight proxy),milking system: Mecánico,ESPAC direct (gl_sistema_ordenio),mixed: direct ESPAC infrastructure + literatur...,50.000,0.0000,...,,0.005333,0.000000,0.00000,0.222222,0.000000,Headcount per kg is based on current inventory...,pasture total = ESPAC pasture + proxy forage +...,infrastructure from ESPAC cattle milking-syste...,FAO small-scale dairy manual - Uses 4 to 6 L w...
1,eggs,chicken,,(unknown),1 kg shell eggs,layer house or coop with drinkers and lighting,literature/system proxy,literature/system proxy,0.222,0.0101,...,,0.000000,3.466728,0.00000,4.750701,0.216135,Egg headcount per kg is based on one laying cy...,pasture total = ESPAC pasture + proxy forage +...,ESPAC livestock modules do not expose direct i...,Producción sostenible en avicultura - Reports ...
2,goat_live,caprine,,(unknown),1 kg meat (carcass-weight proxy),grazing goat holding with shelter and trough,literature/system proxy,literature/system proxy,9.500,0.0000,...,,0.000000,0.000000,0.00000,0.897921,0.000000,Headcount per kg is based on current inventory...,pasture total = ESPAC pasture + proxy forage +...,ESPAC livestock modules do not expose direct i...,FAO-linked livestock watering structures guide...
3,meat_poultry,chicken,,(unknown),1 kg meat (carcass-weight proxy),broiler house or coop for meat birds,literature/system proxy,literature/system proxy,0.275,0.0050,...,,0.000000,0.090667,0.00000,0.183333,0.003333,Headcount per kg is based on current inventory...,pasture total = ESPAC pasture + proxy forage +...,ESPAC livestock modules do not expose direct i...,FAO Rural structures in the tropics - Table 10...
4,milk,cattle,,(unknown),1 kg FPCM,milking system: Mecánico,ESPAC direct (gl_sistema_ordenio),mixed: direct ESPAC infrastructure + literatur...,60.000,0.6000,...,,0.203513,0.000000,0.00000,10.175658,0.101757,Milk headcount per kg is based on lifetime FPC...,pasture total = ESPAC pasture + proxy forage +...,infrastructure from ESPAC cattle milking-syste...,FAO small-scale dairy manual - Uses 4 to 6 L w...
5,ovine_live,ovine,,(unknown),1 kg meat (carcass-weight proxy),grazing ovine holding with shelter and trough,literature/system proxy,literature/system proxy,7.600,0.0000,...,,0.000360,0.000000,0.00000,0.273358,0.000000,Headcount per kg is based on current inventory...,pasture total = ESPAC pasture + proxy forage +...,ESPAC livestock modules do not expose direct i...,FAO-linked livestock watering structures guide...
6,swine_live,swine,,(unknown),1 kg meat (carcass-weight proxy),Ecuador smallholder pigsty or swine pen with f...,literature/system proxy,literature/system proxy,7.000,0.0620,...,,0.015105,0.025175,0.03021,0.140982,0.001249,Headcount per kg is based on current inventory...,pasture total = ESPAC pasture + proxy forage +...,ESPAC livestock modules do not expose direct i...,FAO Livestock housing for pigs - Pig water con...


,metric,value
0,rows,7
1,rows_with_profile,3
2,rows_missing_profile,4
3,rows_with_positive_enteric_ch4,3
4,rows_with_positive_manure_ch4,3
5,rows_with_positive_manure_n2o,3



## Next steps

- Review the exploratory livestock Tier 1 defaults in `inputs/03-05_dfe_factors.yml`, especially productivity-class choices, body-mass proxies, AWMS shares, and crop-versus-manure reporting boundaries.
- After exporting a filtered livestock CSV from notebook 2, rerun this notebook to produce the livestock DFE tables for the same summary strategy.
